# Tutorial 07 — MDO Optimization

Define a multidisciplinary optimization problem, run it with differential evolution,
plot convergence, and inspect which design variables matter most using sensitivity analysis.

**What you'll learn:**
- Set up an `MDOProblem` with design variables, constraints, and an objective
- Run `scipy_de` differential evolution optimizer
- Plot convergence history
- Use `sensitivity()` to rank design variable influence
- Compare baseline vs optimal design side-by-side

In [1]:
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import aerisplane as ap
from aerisplane.core.structures import Spar, TubeSection
from aerisplane.catalog.materials import CFRP_UD
from aerisplane.catalog.motors import sunnysky_x2216_1250
from aerisplane.catalog.batteries import tattu_4s_5200
from aerisplane.catalog.propellers import apc_10x4_7sf
from aerisplane.core.propulsion import ESC, PropulsionSystem

# Wing with CFRP spar
spar = Spar(
    position=0.25,
    section=TubeSection(outer_diameter=0.018, wall_thickness=0.0015),
    material=CFRP_UD,
)
wing = ap.Wing(
    name="main_wing",
    symmetric=True,
    xsecs=[
        ap.WingXSec(xyz_le=[0.00, 0.00, 0.00], chord=0.28,
                    airfoil=ap.Airfoil(name="ag35"), spar=spar),
        ap.WingXSec(xyz_le=[0.07, 1.20, 0.05], chord=0.14, twist=-2.0,
                    airfoil=ap.Airfoil(name="ag35"), spar=spar),
    ],
    control_surfaces=[
        ap.ControlSurface(name="aileron", span_start=0.5, span_end=0.9,
                          chord_fraction=0.25, symmetric=False, max_deflection=20.0),
    ],
)

htail = ap.Wing(
    name="htail",
    symmetric=True,
    xsecs=[
        ap.WingXSec(xyz_le=[0.95, 0.00, 0.00], chord=0.15,
                    airfoil=ap.Airfoil(name="naca0012")),
        ap.WingXSec(xyz_le=[0.98, 0.40, 0.00], chord=0.10,
                    airfoil=ap.Airfoil(name="naca0012")),
    ],
    control_surfaces=[
        ap.ControlSurface(name="elevator", span_start=0.0, span_end=1.0,
                          chord_fraction=0.35, symmetric=True, max_deflection=25.0),
    ],
)

vtail = ap.Wing(
    name="vtail",
    symmetric=False,
    xsecs=[
        ap.WingXSec(xyz_le=[0.90, 0.00, 0.00], chord=0.18,
                    airfoil=ap.Airfoil(name="naca0012")),
        ap.WingXSec(xyz_le=[0.95, 0.00, 0.30], chord=0.12,
                    airfoil=ap.Airfoil(name="naca0012")),
    ],
    control_surfaces=[
        ap.ControlSurface(name="rudder", span_start=0.0, span_end=1.0,
                          chord_fraction=0.30, symmetric=True, max_deflection=25.0),
    ],
)

propulsion = PropulsionSystem(
    motor=sunnysky_x2216_1250,
    propeller=apc_10x4_7sf,
    battery=tattu_4s_5200,
    esc=ESC(name="generic_60A", max_current=60.0, mass=0.025),
)

aircraft = ap.Aircraft(
    name="baseline",
    wings=[wing, htail, vtail],
    propulsion=propulsion,
)

print(f"Baseline span : {wing.span():.2f} m")
print(f"Baseline area : {wing.area():.4f} m²")
print(f"Baseline AR   : {wing.span()**2 / wing.area():.2f}")

Baseline span : 2.40 m
Baseline area : 0.5040 m²
Baseline AR   : 11.43


## Define the mission

We want to minimize total mission energy (equivalent to maximizing endurance).
The mission is a simple cruise + loiter profile.

In [2]:
from aerisplane.mission.segments import Mission, Cruise, Loiter
from aerisplane.mdo import MDOProblem, DesignVar, Constraint, Objective

mission = Mission(
    start_altitude=0.0,
    segments=[
        Cruise(name="cruise_out", velocity=18.0, altitude=100.0, distance=5000.0),
        Loiter(name="loiter",     velocity=16.0, altitude=100.0, duration=600.0),
        Cruise(name="cruise_back",velocity=18.0, altitude=100.0, distance=5000.0),
    ]
)

condition = ap.FlightCondition(velocity=18.0, altitude=100.0)

problem = MDOProblem(
    aircraft=aircraft,
    condition=condition,
    design_variables=[
        DesignVar("wings[0].xsecs[1].chord",     lower=0.08, upper=0.22, scale=0.15),
        DesignVar("wings[0].xsecs[1].xyz_le[1]", lower=0.80, upper=1.80, scale=1.20),
    ],
    constraints=[
        Constraint("structures.wings[0].bending_margin", lower=0.0),
        Constraint("stability.static_margin", lower=0.05, upper=0.25),
    ],
    objective=Objective("mission.total_energy", maximize=False),
    mission=mission,
    alpha=4.0,
)
print("Problem validated. Design variables:")
for dv in problem._dvars:
    print(f"  {dv.path}  [{dv.lower}, {dv.upper}]")

Problem validated. Design variables:
  wings[0].xsecs[1].chord  [0.08, 0.22]
  wings[0].xsecs[1].xyz_le[1]  [0.8, 1.8]


## Run the optimizer

Differential evolution (`scipy_de`) is a population-based global optimizer —
no gradient information needed, robust to local minima.
`maxiter=20` with `popsize=8` is enough for a 2-variable problem to demonstrate convergence.

In [3]:
import logging
logging.basicConfig(level=logging.WARNING)  # suppress INFO logs from the discipline chain

result = problem.optimize(
    method="scipy_de",
    options={"maxiter": 20, "seed": 42, "popsize": 8},
    verbose=False,
)
print(result.report())

/home/stepan/projects/Personal/AerisPlane/.venv/lib/python3.12/site-packages/scipy/optimize/_differentialevolution.py:533: UserWarning: differential evolution didn't find a solution satisfying the constraints, attempting to polish from the least infeasible solution
  ret = solver.solve()


/home/stepan/projects/Personal/AerisPlane/.venv/lib/python3.12/site-packages/scipy/optimize/_trustregion_constr/equality_constrained_sqp.py:217: UserWarning: Singular Jacobian matrix. Using SVD decomposition to perform the factorizations.
  Z, LS, Y = projections(A, factorization_method)


/home/stepan/projects/Personal/AerisPlane/.venv/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:385: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(self.x - self.x_prev, self.g - self.g_prev)
/home/stepan/projects/Personal/AerisPlane/.venv/lib/python3.12/site-packages/scipy/optimize/_differentiable_functions.py:737: UserWarning: delta_grad == 0.0. Check if the approximated function is linear. If the function is linear better results can be obtained by defining the Hessian as zero instead of using quasi-Newton approximations.
  self.H.update(delta_x, delta_g)


AerisPlane Optimisation Result

Objective
----------------------------------------
  Initial  : 216624
  Optimal  : 211528
  Change   : -2.4%
  Evals    : 660
  Constraints satisfied: NO

Design Variables
----------------------------------------
  Path                                             Initial    Optimal
  --------------------------------------------- ---------- ----------
  wings[0].xsecs[1].chord                             0.14    0.21766
  wings[0].xsecs[1].xyz_le[1]                          1.2     1.0201


## Interpreting the result

With a small iteration budget (`maxiter=20`), the optimizer may not satisfy all constraints — this is expected when demonstrating convergence behaviour on a limited budget.

For a production run, increase `maxiter` (e.g. 200–500). The convergence, sensitivity, and comparison cells below work on the best-found solution regardless of feasibility status.

## Convergence history

The optimizer tracks every evaluation. We can plot the best objective value
over evaluations to see how quickly it converged.

In [4]:
history = result.convergence_history
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(history, lw=1.5, color="steelblue")
ax.set_xlabel("Evaluation number")
ax.set_ylabel("Best total energy [J]")
ax.set_title("Convergence history \u2014 scipy_de")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

/tmp/ipykernel_21356/3739005324.py:9: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.show()


## Sensitivity analysis

Which design variable actually matters more? Sensitivity analysis computes
normalized finite-difference gradients at the optimal point.

In [5]:
sens = problem.sensitivity(result.x_optimal)
print(sens.report())

Sensitivity Analysis — Objective Gradients (normalized, ranked)
  Evaluated at x=[1.4511, 0.8501]
  Objective value: 2.1153e+05
  Step size: 1.00e-04

  Objective:
   1. wings[0].xsecs[1].xyz_le[1]                             dObj/dx=+2.357e+05  |norm|=0.9472
   2. wings[0].xsecs[1].chord                                 dObj/dx=+4.928e+04  |norm|=0.338

  Constraint: structures.wings[0].bending_margin
   1. wings[0].xsecs[1].xyz_le[1]                             dCon/dx=+3.731  |norm|=196.2
   2. wings[0].xsecs[1].chord                                 dCon/dx=+0.5777  |norm|=51.84

  Constraint: stability.static_margin
   1. wings[0].xsecs[1].xyz_le[1]                             dCon/dx=+0.4403  |norm|=0.5012
   2. wings[0].xsecs[1].chord                                 dCon/dx=+0.1948  |norm|=0.3786


## Baseline vs optimal comparison

Print a side-by-side comparison of the key design values and discipline results.

In [6]:
print("Design variable comparison:")
print(f"{'Variable':<45}  {'Baseline':>12}  {'Optimal':>12}")
print("-" * 72)
for path, (v0, vopt) in result.variables.items():
    print(f"{path:<45}  {v0:>12.4f}  {vopt:>12.4f}")

print(f"\nObjective (total energy [J]): {result.objective_initial:.1f} \u2192 {result.objective_optimal:.1f}")
print(f"Improvement: {(result.objective_initial - result.objective_optimal)/result.objective_initial:.1%}")
print(f"Constraints satisfied: {result.constraints_satisfied}")
print(f"Total evaluations: {result.n_evaluations}")

Design variable comparison:
Variable                                           Baseline       Optimal
------------------------------------------------------------------------
wings[0].xsecs[1].chord                              0.1400        0.2177
wings[0].xsecs[1].xyz_le[1]                          1.2000        1.0201

Objective (total energy [J]): 216623.5 → 211527.7
Improvement: 2.4%
Constraints satisfied: False
Total evaluations: 660
